# Financial News ABSA Pipeline

End-to-end Aspect-Based Sentiment Analysis on financial headlines:
1. **NER** — extract company/entity mentions (`ner_bert_final`)
2. **ABSA** — classify sentiment per entity (`absa_finbert_final`)

**Before running:** upload your fine-tuned model folders to Google Drive at:
```
MyDrive/TextMining/models/ner_bert_final/
MyDrive/TextMining/models/absa_finbert_final/
```
These are saved by `04_transformer_models.ipynb` into `backend/models/`.

In [3]:
# !pip install -q transformers accelerate

In [4]:
# from google.colab import drive
# drive.mount('/content/drive')

In [8]:
from pathlib import Path

PROJECT_ROOT   = Path('..').resolve()
MODEL_DIR      = PROJECT_ROOT / 'backend/models'

assert (MODEL_DIR / 'ner_bert_final').exists(), \
    f"NER model not found at {MODEL_DIR / 'ner_bert_final'}"
assert (MODEL_DIR / 'absa_finbert_final').exists(), \
    f"ABSA model not found at {MODEL_DIR / 'absa_finbert_final'}"

In [9]:
import numpy as np
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
)

NER_ID2LABEL  = {0: "O", 1: "B-ENT", 2: "I-ENT"}
ABSA_ID2LABEL = {0: "negative", 1: "neutral", 2: "positive"}

/home/anton/TextMining/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
class ABSAPipeline:
    def __init__(self, device: str | None = None):
        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"
        self.device = device

        ner_path = MODEL_DIR / "ner_bert_final"
        self.ner_tokenizer = AutoTokenizer.from_pretrained(ner_path)
        self.ner_model = (
            AutoModelForTokenClassification.from_pretrained(ner_path).to(device).eval()
        )

        absa_path = MODEL_DIR / "absa_finbert_final"
        self.absa_tokenizer = AutoTokenizer.from_pretrained(absa_path)
        self.absa_model = (
            AutoModelForSequenceClassification.from_pretrained(absa_path).to(device).eval()
        )

    def extract_entities(self, headline: str) -> list[str]:
        words = headline.split()
        enc = self.ner_tokenizer(
            words,
            is_split_into_words=True,
            return_tensors="pt",
            truncation=True,
            max_length=128,
        ).to(self.device)

        with torch.no_grad():
            logits = self.ner_model(**enc).logits

        pred_ids = logits[0].argmax(-1).cpu().tolist()
        word_ids = enc.word_ids(batch_index=0)

        word_tags: dict[int, str] = {}
        for tok_idx, wid in enumerate(word_ids):
            if wid is not None and wid not in word_tags:
                word_tags[wid] = NER_ID2LABEL[pred_ids[tok_idx]]

        entities: list[str] = []
        current: list[str] = []
        for wid in sorted(word_tags):
            tag = word_tags[wid]
            if tag == "B-ENT":
                if current:
                    entities.append(" ".join(current))
                current = [words[wid]]
            elif tag == "I-ENT" and current:
                current.append(words[wid])
            else:
                if current:
                    entities.append(" ".join(current))
                    current = []
        if current:
            entities.append(" ".join(current))

        return entities

    def classify_sentiment(self, headline: str, entity: str) -> dict:
        enc = self.absa_tokenizer(
            headline,
            entity,
            return_tensors="pt",
            truncation=True,
            max_length=128,
        ).to(self.device)

        with torch.no_grad():
            logits = self.absa_model(**enc).logits

        probs = torch.softmax(logits[0], dim=-1).cpu().tolist()
        pred_id = int(np.argmax(probs))

        return {
            "sentiment": ABSA_ID2LABEL[pred_id],
            "confidence": round(probs[pred_id], 4),
            "scores": {ABSA_ID2LABEL[i]: round(p, 4) for i, p in enumerate(probs)},
        }

    def analyze(self, headline: str) -> dict:
        entities = self.extract_entities(headline)
        results = [
            {"entity": ent, **self.classify_sentiment(headline, ent)}
            for ent in entities
        ]
        return {"headline": headline, "entities": results}

    def analyze_batch(self, headlines: list[str]) -> list[dict]:
        return [self.analyze(h) for h in headlines]

In [11]:
pipe = ABSAPipeline()
print(f"Device: {pipe.device}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2510.20it/s]


Device: cuda


In [12]:
samples = [
    "YES Bank shares surge after RBI clears rescue plan",
    "Infosys Q3 profit falls short of estimates; TCS reports record earnings",
    "Trade long on infrastructure stocks: Devang Visaria",
    "MMTC is bad",
]

for headline in samples:
    result = pipe.analyze(headline)
    print(f"Headline : {result['headline']}")
    if result["entities"]:
        for e in result["entities"]:
            print(f"  [{e['sentiment']:8s} {e['confidence']:.2f}]  {e['entity']}")
    else:
        print("  (no entities detected)")
    print()

Headline : YES Bank shares surge after RBI clears rescue plan
  [positive 1.00]  YES Bank

Headline : Infosys Q3 profit falls short of estimates; TCS reports record earnings
  [negative 1.00]  Infosys
  [negative 0.96]  TCS

Headline : Trade long on infrastructure stocks: Devang Visaria
  [positive 0.61]  infrastructure stocks:

Headline : MMTC is bad
  [negative 0.93]  MMTC



In [14]:
headline = "Reliance Industries posts record quarterly profit on telecom surge"  # @param {type:"string"}

result = pipe.analyze(headline)
print(f"Headline : {result['headline']}")
if result["entities"]:
    for e in result["entities"]:
        print(f"  [{e['sentiment']:8s} {e['confidence']:.2f}]  {e['entity']}")
        print(f"    scores: {e['scores']}")
else:
    print("  (no entities detected)")

Headline : Reliance Industries posts record quarterly profit on telecom surge
  [positive 1.00]  Reliance Industries
    scores: {'negative': 0.0008, 'neutral': 0.0033, 'positive': 0.9959}


In [22]:
#Multiple headlines batch demo
headlines = [
    "YES Bank shares surge after RBI clears rescue plan",
    "Infosys Q3 profit falls short of estimates; TCS reports record earnings",
    "Trade long on infrastructure stocks: Devang Visaria",
    "MMTC is bad",
    # add multi conflicting sentiments
    "Microchip posts record quarterly profit on telecom surge while Apple callable bonds are downgraded by S&P",
]
results = pipe.analyze_batch(headlines)
for res in results:
    print(f"Headline : {res['headline']}")
    if res["entities"]:
        for e in res["entities"]:
            print(f"  [{e['sentiment']:8s} {e['confidence']:.2f}]  {e['entity']}")
    else:
        print("  (no entities detected)")
    print()

Headline : YES Bank shares surge after RBI clears rescue plan
  [positive 1.00]  YES Bank

Headline : Infosys Q3 profit falls short of estimates; TCS reports record earnings
  [negative 1.00]  Infosys
  [negative 0.96]  TCS

Headline : Trade long on infrastructure stocks: Devang Visaria
  [positive 0.61]  infrastructure stocks:

Headline : MMTC is bad
  [negative 0.93]  MMTC

Headline : Microchip posts record quarterly profit on telecom surge while Apple callable bonds are downgraded by S&P
  [positive 1.00]  Microchip
  [negative 0.96]  Apple

